<div class="jcs-title"><div class="jcs-kicker">Human–AI Commons · JCS · 01</div>
<h1>From a resource request to a shared environment</h1>
<p>A mathematical and interactive introduction to the journal model.</p></div>

An actor can lower a resource request today and still face scarcity tomorrow. Someone else may benefit from the water left behind, while renewed abundance can make restraint less attractive. This notebook follows that chain from **one decision**, through **one resource budget**, to **a small spatial world**.

The figures use the same Python model as the study. The small examples here illustrate its rules; they are not additional research evidence. No production bundle is needed.

**Before opening:** complete the [JCS quick start](../../README.md#jcs-quick-start) once. From the repository root, launch `.venv/bin/jupyter lab jcs/notebooks/` and select its **Python 3** kernel. Run cells in order; sliders and selectors update their figures. [Load and local interventions](03_load_and_interventions.ipynb) follows the paper's results, and [Exploring saved worlds](02_exploring_saved_worlds.ipynb) reads saved results of the earlier staged payoff search.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML

%matplotlib inline

here = Path.cwd().resolve()
JCS_ROOT = next((p for p in [here, *here.parents] if (p / "src/human_ai_commons").is_dir()), None)
if JCS_ROOT is None and (here / "jcs/src/human_ai_commons").is_dir():
    JCS_ROOT = here / "jcs"
if JCS_ROOT is None:
    raise RuntimeError("Open the notebook from the JCS folder or its parent repository.")
for path in [JCS_ROOT / "src", JCS_ROOT / "notebooks"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
from _explore import style, BLUE, TEAL, ORANGE, GRAY, frame_map, map_legend
style()
display(HTML("""<style>
.jcs-title {border-left:5px solid #278477;padding:12px 22px;background:#f3f7f7;margin:16px 0 24px}
.jcs-title h1 {margin:4px 0 12px;font-weight:600;color:#244867}
.jcs-kicker {font-size:12px;letter-spacing:2px;color:#278477;text-transform:uppercase}
.jcs-note {border-left:3px solid #bc703a;padding:10px 16px;background:#fbf7f2;margin:16px 0}
</style>"""))


## 1 · Two requests, one fixed target

Cooperation, $C$, means a **lower request** $c_i$. Defection, $D$, means a **higher request** $d_i=1.5c_i$. These are game labels, not judgments about the value of an activity. Both agent types follow the same rule; their resource demands differ.

The share of demand met compares delivery $y_i$ with the higher request $d_i$, the agent's demand:

$$s_i=\min\!\left(\frac{y_i}{d_i},1\right).$$

Receiving the lower request in full therefore meets **two thirds of the agent's demand**. The share of requests delivered asks a different question: did the agent receive what it actually requested? Keeping these denominators separate makes the cost of restraint visible.


In [ ]:
from human_ai_commons.config import load_config, scenario, metric_spec, welfare_spec, world_spec
from human_ai_commons.payoffs import Policy
from human_ai_commons.dynamics import multiplier, Simulator, Drivers, simulate
from human_ai_commons.world import World, WorldSpec, generate_world
from dataclasses import replace

config = load_config(JCS_ROOT / "configs/research_protocol.json")
ratio = config["world"]["defect_to_cooperate_request_ratio"]
display(pd.DataFrame({
    "Choice": ["Lower request (C)", "Higher request (D)"],
    "Request / cooperative request": [1, ratio],
    "Activity target met if fully delivered (%)": [100 / ratio, 100],
}))


## 2 · When does restraint become attractive?

Let $n\in[0,1]$ be the selected source's stock divided by its capacity, **after renewal and before withdrawal**. Let $x\in[0,1]$ be the share of observed other actors making lower requests at the previous step. Rows and columns of the payoff matrix are ordered $C,D$:

$$A_{u,v}(n)=(1-n)\begin{pmatrix}t+u&v\\1&s\end{pmatrix}
+n\begin{pmatrix}1&s\\t&0\end{pmatrix},\qquad t=2,\ s=-\tfrac12.$$

- $u$ changes the payoff from restraint **alongside others who restrain**.
- $v$ changes it **when others continue requesting more**.

A participation bonus and support for restraint despite incomplete participation illustrate these two social meanings. The model changes mathematical incentives; it does not implement payments or a governance institution. Both changes fade with abundance through $1-n$.

The advantage of the higher request is

$$G(n,x)=(2n-1)\underbrace{[x(t-1)+(1-x)(-s)]}_{a(x)}
-(1-n)\underbrace{[xu+(1-x)v]}_{b(x)}.$$

The lower request is preferred when $G\leq0$, including a tie. Its switching level is

$$n^*(x)=\frac{a(x)+b(x)}{2a(x)+b(x)}.$$

At $u=v=0$, that level is one-half for every $x$: abundance above half capacity favors the higher request by construction. Independent action reversal with probability $\varepsilon=0.1$ turns a preferred choice into a 90% or 10% chance of restraint, rather than an absolute rule.


In [ ]:
original = Policy()
x_check = np.linspace(0, 1, 11)
assert np.allclose(original.gap(.5, x_check), 0)
assert np.all(original.gap(1, x_check) > 0)
assert np.all(original.gap(0, x_check) < 0)
print("Original game: lower requests are preferred at or below half capacity.")


### Move the decision boundary

The left panel measures the advantage of requesting more. The right shows which request is preferred at each combination of stock and participation. The dot is the state selected by the controls. The shaded region is an **analytical decision rule**, not an interpolated simulation result.

A positive $u$ can make restraint attractive at higher stocks when participation is widespread. A positive $v$ can make it attractive at higher stocks when participation is low. Their relative changes are $r_C=u/(t-1)$ and $r_D=v/(-s)$. When $r_C=r_D$, participation drops out of the switching level.


In [ ]:
def show_decision(u=0., v=0., n=.55, x=.5):
    policy = Policy(u=u, v=v)
    stock = np.linspace(0, 1, 301)
    participation = np.linspace(0, 1, 201)
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.7), layout="constrained")
    for fraction, color, label in [(0, GRAY, "Others request more"), (x, BLUE, "Selected participation"), (1, TEAL, "Others restrain")]:
        axes[0].plot(stock, policy.gap(stock, fraction), color=color, label=label)
    axes[0].axhline(0, color="black", lw=.7)
    axes[0].scatter([n], [policy.gap(n, x)], color=ORANGE, s=55, zorder=4)
    axes[0].set(xlabel="Stock / capacity", ylabel="Advantage of the higher request", title="A choice changes at zero")
    axes[0].legend(frameon=False, fontsize=8)
    nn, xx = np.meshgrid(stock, participation)
    gap = policy.gap(nn, xx)
    axes[1].contourf(nn, xx, gap <= 0, levels=[-.5, .5, 1.5], colors=["#f1dcc9", "#d4e8e4"])
    axes[1].contour(nn, xx, gap, levels=[0], colors=[BLUE], linewidths=1.5)
    axes[1].scatter([n], [x], color=ORANGE, edgecolor="white", s=65, zorder=4)
    axes[1].text(.03, .93, "Lower request preferred", transform=axes[1].transAxes, color=BLUE, fontsize=9)
    axes[1].text(.55, .06, "Higher request preferred", transform=axes[1].transAxes, color="#87502c", fontsize=9)
    axes[1].set(xlabel="Stock / capacity", ylabel="Others making lower requests", title="The rule across local states")
    plt.show(); plt.close(fig)
    a = x * (policy.t - 1) + (1 - x) * (-policy.s)
    b = x * u + (1 - x) * v
    threshold = (a + b) / (2 * a + b)
    assert abs(policy.gap(threshold, x)) < 1e-12
    preferred = policy.gap(n, x) <= 0
    probability = 1 - config["world"]["action_reversal_probability"] if preferred else config["world"]["action_reversal_probability"]
    display(Markdown(f"At **{n:.0%} stock** and **{x:.0%} participation**, the preferred choice is **{'lower' if preferred else 'higher'}**. "
                     f"The switching stock is **{threshold:.1%}**; reversal makes the probability of a lower request **{probability:.0%}**. "
                     f"Relative changes: $r_C={u:.3f}$, $r_D={2*v:.3f}$. Payoff-change cost: **{policy.cost():.3f}** "
                     f"({'within' if policy.cost() <= .5 else 'above'} the primary budget 0.5)."))

u_control = widgets.FloatSlider(value=0, min=-.9, max=1, step=.01, description="u: alongside restraint", continuous_update=False, style={"description_width": "170px"}, layout=widgets.Layout(width="470px"))
v_control = widgets.FloatSlider(value=0, min=-.45, max=.8, step=.01, description="v: despite higher requests", continuous_update=False, style={"description_width": "170px"}, layout=widgets.Layout(width="470px"))
n_control = widgets.FloatSlider(value=.55, min=0, max=1, step=.01, description="Stock / capacity", continuous_update=False, style={"description_width": "170px"}, layout=widgets.Layout(width="470px"))
x_control = widgets.FloatSlider(value=.5, min=0, max=1, step=.01, description="Lower-request share x", continuous_update=False, style={"description_width": "170px"}, layout=widgets.Layout(width="470px"))
presets = {"Original": (0., 0.), "Earlier restraint": (.497265625, .2486328125),
           "Participation-dependent": (.2484375, -.06015625), "Offsetting changes": (.117350147545, -.117350147545),
           "Later restraint": (-.95, -.475)}
# the later-restraint example needs the full permitted negative bounds.
u_control.min = -.95; v_control.min = -.475
preset_control = widgets.Dropdown(options=list(presets), description="Example")
def choose_example(change):
    u_control.value, v_control.value = presets[change["new"]]
preset_control.observe(choose_example, names="value")
decision_output = widgets.interactive_output(show_decision, {"u": u_control, "v": v_control, "n": n_control, "x": x_control})
display(preset_control, widgets.VBox([u_control, v_control, n_control, x_control]), decision_output)


<div class="jcs-note"><strong>A useful comparison.</strong> At stock 0.55, the original game prefers the higher request. Select earlier restraint: the preferred action changes, even though the physical stock is identical. Select offsetting changes and vary participation: increasing one payoff while decreasing the other can matter locally, despite cancelling at half participation.</div>

These controls explore admissible endpoint games. A selected combination may exceed the primary search budget; the displayed cost makes that distinction explicit. The named examples also use different actual costs and, for later restraint, a larger permitted budget.

## 3 · A decision is followed by a physical balance

Renewal depends on the **previous**, post-withdrawal stock $n=z/K$ and previous nearby cooperation $x$:

$$m=\operatorname{clip}\!\left(1+\tfrac12n(1-n)(\theta x-1),1,3\right),\quad
\widetilde z=\min(K,z+b\,m),\quad z_{\mathrm{next}}=\widetilde z-W.$$

Here $K$ is capacity, $b$ baseline renewal, $\theta=3$ the feedback strength, $m$ the renewal multiplier, and $W$ the water actually withdrawn. Actual renewal is $\widetilde z-z$; potential renewal above capacity cannot enter storage. The multiplier's floor is one in the journal reference implementation.

During proportional allocation, total requests $Q$ receive the common fraction $f=\min(1,\widetilde z/Q)$. Each delivery is $y_i=fq_i$, where $q_i$ is that actor's current request. The next cell evaluates one controlled source using `Simulator.step`, with human and AI demand in the study's 1:20 ratio. It forces both actors to the chosen request only to isolate the physical accounting. This is a constructed one-step example, not a production initialization.


In [ ]:
one_source = World(
    WorldSpec(width=5, height=5, humans=1, ai=1, density=1., capacity=20.),
    np.array([[2, 2]]), np.array([20.]), np.array([.3]),
    np.array([[2, 2], [2, 3]]), np.array([False, True]), np.array([True, True]),
)
def show_flow(n=.03, pressure=2., action="C"):
    physical = scenario(config, pressure)
    engine = Simulator(one_source, physical, Policy(forced=action))
    engine.water[:] = n * one_source.capacity
    start = engine.water.sum()
    event = engine.step(np.array([.2, .8]), np.array([.7, .7]), np.array([0, 1]))
    renewed = event["resource_renewal"].sum()
    withdrawn = event["source_withdrawal"].sum()
    end = engine.water.sum()
    assert np.isclose(start + renewed - withdrawn, end)
    target = engine.cooperative_request * physical.defect_ratio
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), layout="constrained")
    axes[0].bar(["Starting\nstock", "Actual\nrenewal", "Withdrawn", "Remaining"], [start, renewed, -withdrawn, end], color=[GRAY, TEAL, ORANGE, BLUE])
    axes[0].axhline(0, color="black", lw=.7)
    axes[0].set(title="Water is conserved", ylabel="Resource units")
    x = np.arange(2)
    axes[1].bar(x - .18, 100 * event["requested"] / target, .34, label="Requested", color=GRAY)
    axes[1].bar(x + .18, 100 * event["delivered"] / target, .34, label="Delivered", color=TEAL)
    axes[1].set(xticks=x, xticklabels=["Human", "AI"], ylim=(0, 110), ylabel="Share of own fixed target (%)", title="Requests and deliveries / fixed target")
    axes[1].legend(frameon=False)
    plt.show(); plt.close(fig)
    display(Markdown(f"**{start:.3f} + {renewed:.3f} − {withdrawn:.3f} = {end:.3f}** resource units. "
                     f"Both actors make the **{'lower' if action == 'C' else 'higher'} request**. "
                     "The target stays fixed when their request changes. Under proportional rationing, both receive the same fraction of their request at this source."))

flow_stock = widgets.FloatSlider(value=.03, min=0, max=1, step=.01, description="Starting stock", continuous_update=False)
flow_pressure = widgets.FloatSlider(value=2, min=.25, max=6, step=.25, description="Demand pressure", continuous_update=False, style={"description_width": "initial"})
flow_action = widgets.ToggleButtons(options=[("Both request less", "C"), ("Both request more", "D")], description="Action")
flow_output = widgets.interactive_output(show_flow, {"n": flow_stock, "pressure": flow_pressure, "action": flow_action})
display(widgets.VBox([flow_stock, flow_pressure, flow_action]), flow_output)


At full stock, renewal may be clipped even though the renewal law offers a positive input. At an empty source, incoming renewal can be withdrawn immediately. When requests already exceed available stock, lowering requests need not lower the total withdrawal: the source can still be exhausted.

The first figure measures **water volume**. The second compares each actor with **its own activity target**. Equal coverage percentages can coexist with unequal water volumes.

## 4 · Why space changes the story

A source has a small set of direct users within access radius one. Its renewal responds to actors within radius three. Payoff decisions also use composition around the selected source, excluding the focal actor. Positions are fixed, but an actor samples an accessible source at each step. The next observation can therefore concern a different source from the one just used.

One model step proceeds as follows:

**Previous stock and actions → renewal → source selection → planned actions → simultaneous allocation → recorded stock and actions.**

The grid wraps at its edges: nearby cells across a boundary are neighbors. That geometry is a torus. There is no learning or explicit delayed perception rule; longer responses arise from coupled states and spatial relationships.

The next example uses a 12 × 10 world, 8 agents of each type and 160 steps. All other physical rules come from the research configuration. Each design receives the same world and indexed random inputs. The first 40 steps are shaded only to illustrate the distinction between an initial period and a measurement window. These settings differ from the production experiment.


In [ ]:
small_spec = replace(world_spec(config), width=12, height=10, humans=8, ai=8)
small_world = generate_world(small_spec, 42)
small_policies = {"Original": Policy(), "Earlier restraint": Policy(u=.497265625, v=.2486328125),
                  "Participation-dependent": Policy(u=.2484375, v=-.06015625)}
small_runs = {}
for label, policy in small_policies.items():
    summary, history = simulate(small_world, scenario(config, 1), policy, 2026, 160, 40,
                                metric_spec(config), record=True, welfare=welfare_spec(config))
    small_runs[label] = (summary, history)
print(f"Illustrative world: {len(small_world.resources)} sources, {len(small_world.positions)} actors; 3 matched 160-step histories.")


In [ ]:
def show_world(design="Original", step=40):
    history = small_runs[design][1]
    original_history = small_runs["Original"][1]
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), layout="constrained")
    image = frame_map(axes[0], small_world, history["stocks"][step-1], history["strategies"][step-1], f"{design} · step {step}")
    fig.colorbar(image, ax=axes[0], shrink=.7, label="Stock / capacity")
    paths = [(original_history, "Original", GRAY)] if design == "Original" else [(original_history, "Original", GRAY), (history, design, BLUE)]
    for data, label, color in paths:
        mean_stock = (data["stocks"][:, data["accessible"]] / small_world.capacity[data["accessible"]]).mean(axis=1)
        axes[1].plot(np.arange(1, 161), mean_stock, color=color, label=label)
        axes[2].plot(np.arange(1, 161), 100 * data["strategies"].mean(axis=1), color=color, label=label)
    for ax in axes[1:]:
        ax.axvspan(1, 40, color=GRAY, alpha=.12)
        ax.axvline(step, color=ORANGE, lw=1)
        ax.set_xlabel("Recorded step")
    axes[1].set(title="Accessible resources", ylabel="Mean stock / capacity", ylim=(0, 1.03))
    axes[2].set(title="Lower requests", ylabel="Actors making lower requests (%)", ylim=(0, 100))
    axes[1].legend(frameon=False, fontsize=8)
    fig.legend(handles=map_legend(), loc="outside lower center", ncol=4, frameon=False, fontsize=9)
    plt.show(); plt.close(fig)

world_design = widgets.Dropdown(options=list(small_runs), description="Design")
world_step = widgets.IntSlider(value=40, min=1, max=160, description="Recorded step", continuous_update=False)
world_output = widgets.interactive_output(show_world, {"design": world_design, "step": world_step})
display(widgets.HBox([world_design, world_step]), world_output)


Resource squares are shaded by stock. Circles represent human actors and triangles AI actors; blue actors make lower requests and orange actors higher requests. Slight marker offsets separate actors sharing a cell. Blank cells contain no source. These are post-withdrawal states; an actor's decision used the source's earlier replenished stock.

A shared mean can hide different local burdens. Conversely, a striking map from one world cannot establish an ensemble result. Matched worlds isolate payoff comparisons within a given realization; independent worlds determine how broadly that comparison holds.

## 5 · From the example to the experiment

| Measurement | What it answers | What it does not establish |
|---|---|---|
| Lower-request frequency | How often agents restrain their requests | How much of their demand is met |
| Share of demand met, for human and for AI agents | How much of each group's demand is delivered | Overall welfare or the social value of either activity |
| Share of requests delivered | Whether a group received what it asked for | Whether it asked for less |
| Share of sources nearly empty | How many sources hold less than 10% of their capacity on average | Why they are empty; the load on each source answers that |

The paper adds the expected load of each source and a delivery limit computed from positions alone; [Load and local interventions](03_load_and_interventions.ipynb) shows both. The earlier staged payoff search, which [Exploring saved worlds](02_exploring_saved_worlds.ipynb) explores, scored depletion at the worst source histories and tested a prolonged-low-stock condition instead; the paper uses neither.

The model's abundance response is prescribed by the payoff rule. Demonstrating a harmful, self-maintaining spatial feedback process requires more than finding low stocks or observing a temporary change in cooperation.

**Next:** [Load and local interventions](03_load_and_interventions.ipynb) follows the paper from the load on each source to the three interventions. [Exploring saved worlds](02_exploring_saved_worlds.ipynb) reads results of the earlier staged payoff search. The definitions are in the [experiment descriptions](../docs/experiments.md) and the [implementation reference](../docs/implementation.md).